# Gemini finetune test

In [ ]:
import os
import json

In [10]:
def clean_data(data):
    # Your cleaning logic here
    # Clean sentences (Keep all entries within quotes ‘’), if not, keep the whole
    cleaned_data = []
    for entry in data:
        if entry == None:
            continue
        original = entry['source']
        translated = entry['translation']

        # Extract text within quotes if present
        if '‘' in original and '’' in original:
            start = original.index('‘') + 1
            end = original.index('’')
            original = original[start:end].strip()
        
        if '‘' in translated and '’' in translated:
            start = translated.index('‘') + 1
            end = translated.index('’')
            translated = translated[start:end].strip()

        # If "Translation: " prefix exists, remove it
        if translated.lower().startswith("translation:"):
            translated = translated[len("translation:"):].strip()
        
        if translated.lower().startswith("translation -"):
            translated = translated[len("translation -"):].strip()
        if translated.lower().startswith("translation in english:"):
            translated = translated[len("translation in english:"):].strip()

        # Do it for all other types of quotes as well
        quote_pairs = [('“', '”'), ('‘', '’'), ('"', '"'), ("'", "'")]
        for open_quote, close_quote in quote_pairs:
            if translated.startswith(open_quote) and translated.endswith(close_quote):
                translated = translated[1:-1].strip()
            if original.startswith(open_quote) and original.endswith(close_quote):
                original = original[1:-1].strip()
        cleaned_data.append({
            'source': original,
            'translation': translated
        })


    return cleaned_data

In [ ]:
def get_gemini_finetune_json(train_data, test_data, base_filename, language_name):
    
    output_path = "../data/gemini_finetune_data"

    combined_output_path = os.path.join(output_path, language_name)
    os.makedirs(combined_output_path, exist_ok=True)
    output_filename = os.path.join(combined_output_path, base_filename + "_train.jsonl")

    with open(output_filename, "w", encoding="utf-8") as outfile:
        for entry in train_data:
            json_line = {
                "contents":[
                    {
                        "role": "user",
                        "parts": [
                            {"text": f"Translate to english from {language_name}: " + entry["source"]}
                        ]
                    },
                    {

                        "role": "model",
                        "parts": [
                            {"text": entry["translation"]}
                        ],
                    }
                ]
            }
            outfile.write(json.dumps(json_line) + "\n")

    print(f"Saved {len(train_data)} training entries to {output_filename}")

    output_filename = os.path.join(combined_output_path, base_filename + "_test.jsonl")
    with open(output_filename, "w", encoding="utf-8") as outfile:
        for entry in test_data:
            json_line = {
                "contents":[
                    {
                        "role": "user",
                        "parts": [
                            {"text": f"Translate to english from {language_name}: " + entry["source"]}
                        ]
                    },
                    {

                        "role": "model",
                        "parts": [
                            {"text": entry["translation"]}
                        ],
                    }
                ]
            }
            outfile.write(json.dumps(json_line) + "\n")
    print(f"Saved {len(test_data)} testing entries to {output_filename}")

In [23]:
language_name = "kalamang"
file_dir = f"../data/generated_results_batched/{language_name}"

for filename in os.listdir(file_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(file_dir, filename)
        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        
        base_filename = filename.split(".json")[0]
        cleaned_data = clean_data(data)

        get_gemini_finetune_json(cleaned_data, cleaned_data, base_filename, language_name)

        print(f"File: {base_filename}, Number of entries: {len(data)}")

Saved 3557 training entries to ../data/gemini_finetune_data\kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10_train.jsonl
Saved 3557 testing entries to ../data/gemini_finetune_data\kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10_test.jsonl
File: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10, Number of entries: 3557
Saved 5335 training entries to ../data/gemini_finetune_data\kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15_train.jsonl
Saved 5335 testing entries to ../data/gemini_finetune_data\kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15_test.jsonl
File: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_15, Number of entries: 5335
Saved 7113 training entries to ../data/gemini_finetune_data\kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20_train.jsonl
Saved 7113 testing entries to ../data/gemini_finetune_data\kalamang\kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_20_test.jsonl
File: kalamang_all_batch_prompts_A

In [40]:
import json

In [41]:
# Create test set using original parallel sentences after 400
language_name = "kalamang"
parallel_sents_file = f"../data/parallel_sentences/{language_name}_test_set_cleaned.json"

if os.path.exists(parallel_sents_file):
    with open(parallel_sents_file, "r", encoding="utf-8") as file:
        parallel_data = json.load(file)
    test_set = parallel_data
    print(f"Created test set with {len(test_set)} entries from parallel sentences.")
else:
    print(f"File {parallel_sents_file} does not exist.")

Created test set with 500 entries from parallel sentences.


In [42]:
test_set_sents_only = [
    {
        "source": entry["source"],
        "translation": entry["translation"]
    }
    for entry in test_set
]

In [26]:
test_set_sents_only[0], len(test_set_sents_only)

({'source': 'ma keit osangga puru =ten=kap=te bara',
  'translation': 'He fell from that top up there.'},
 500)

In [44]:
# save as test.jsonl
output_dir = os.path.join("../data/gemini_finetune_data", language_name)
output_filename = os.path.join(output_dir, "test.jsonl")
with open(output_filename, "w", encoding="utf-8") as outfile:  
    for entry in test_set_sents_only:
        json_line = {
            "contents":[
                {
                    "role": "user",
                    "parts": [
                        {"text": f"Translate to english from {language_name}: " + entry["source"]}
                    ]
                },
                {

                    "role": "model",
                    "parts": [
                        {"text": entry["translation"]}
                    ],
                }
            ]
        }
        outfile.write(json.dumps(json_line) + "\n")
print(f"Saved {len(test_set_sents_only)} testing entries to {output_filename}")





Saved 500 testing entries to ../data/gemini_finetune_data\kalamang\test.jsonl


In [48]:
# Create test set for inference

inference_output_filename = os.path.join(output_dir, f"test_inference.jsonl")

with open(inference_output_filename, "w", encoding="utf-8") as outfile:
    for i, entry in enumerate(test_set_sents_only):
        json_line = {
            # "key" helps you re-sort the results later
            "key": f"{language_name}_sent_{i}", 
            "request": {
                "contents": [
                    {
                        "role": "user",
                        "parts": [
                            {"text": f"Translate to english from {language_name}: " + entry["source"]}
                        ]
                    }
                ]
            }
        }
        outfile.write(json.dumps(json_line) + "\n")

print(f"Saving {len(test_set_sents_only)} inference entries to {inference_output_filename}")

Saving 500 inference entries to ../data/gemini_finetune_data\kalamang\test_inference.jsonl


# SDK Finetuning test

In [ ]:
import time
from vertexai.tuning import sft
from google.cloud import aiplatform
from google.cloud import storage
from google.oauth2 import service_account
import vertexai
import os


: 

In [ ]:
project_id = "project-128068bc-0c84-4cfe-944"
location = "europe-west1"
bucket_name = "thesis_finetune_bucket_1"

In [ ]:
language_name = "kalamang"
data_folder = f"../data/gemini_finetune_data/{language_name}"


train_files = []
test_file = None
for file in os.listdir(data_folder):
    print(file)
    if file.endswith("_train.jsonl"):
        train_files.append(os.path.join(data_folder, file))
    elif file == "test.jsonl":
        test_file = os.path.join(data_folder, file)
    elif file == "test_inference.jsonl":
        test_inference_file = os.path.join(data_folder, file)
print(f"Number of training files: {len(train_files)}")
print(f"Number of testing files: {1 if test_file else 0}")
print(f"Number of testing inference files: {1 if test_inference_file else 0}")

: 

In [18]:
service_account_key_file = "../api_keys/service_account_keys/service_account_key.json"

print(os.path.exists(service_account_key_file))

True


In [19]:
creds = service_account.Credentials.from_service_account_file(service_account_key_file)

In [21]:
vertexai.init(project=project_id, location=location,credentials=creds)
storage_client = storage.Client(credentials=creds,project=project_id)
bucket = storage_client.bucket(bucket_name)

In [22]:
def upload_to_gcs(local_path, gcs_path):
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local_path)
    return f"gs://{bucket.name}/{gcs_path}"



In [23]:
# 1. Prepare common test set
test_set_uri = upload_to_gcs(test_file, f"{language_name}/tuning/test.jsonl")

print(f"Uploaded test set to {test_set_uri}")

Uploaded test set to gs://thesis_finetune_bucket_1/kalamang/tuning/test.jsonl


In [52]:
test_set_inference_uri = upload_to_gcs(test_inference_file,f"{language_name}/tuning/test_inference.jsonl")

print(f"Uploaded test inference file to {test_set_inference_uri}")

Uploaded test inference file to gs://thesis_finetune_bucket_1/kalamang/tuning/test_inference.jsonl


In [33]:
import os
import time
import pandas as pd
from vertexai.tuning import sft

In [34]:
project_id = "project-128068bc-0c84-4cfe-944"
location = "europe-west1"
bucket_name = "thesis_finetune_bucket_1"

In [35]:
service_account_key_file = "../api_keys/service_account_keys/service_account_key.json"

print(os.path.exists(service_account_key_file))

True


In [36]:
creds = service_account.Credentials.from_service_account_file(service_account_key_file)

In [37]:
vertexai.init(project=project_id, location=location,credentials=creds)
storage_client = storage.Client(credentials=creds,project=project_id)
bucket = storage_client.bucket(bucket_name)

In [ ]:
import os
import time
import pandas as pd
from vertexai.tuning import sft
from google.cloud import aiplatform
from vertexai.generative_ai.evaluation import EvalTask

# --- CONFIGURATION ---
MAX_CONCURRENT_JOBS = 1  
TRACKING_FILE = f"{language_name}_tuning_experiments.csv"
TEST_SET_GCS_INFERENCE = test_inference_file

def get_active_jobs_count():
    jobs = sft.SupervisedTuningJob.list()
    active_states = ["JOB_STATE_RUNNING", "JOB_STATE_PENDING", "JOB_STATE_QUEUED"]
    return len([j for j in jobs if j.state.name in active_states])

for file in train_files[:1]:  # Limiting to first 1 file for testing
    base_filename = os.path.basename(file).split("_train.jsonl")[0]
    print(f"Processing file: {base_filename}")
    
    # 1. Skip if already done
    df_tracking = pd.read_csv(TRACKING_FILE)
    if not df_tracking[df_tracking['batch_name'] == base_filename].empty:
        if df_tracking.loc[df_tracking['batch_name'] == base_filename, 'status'].values[0] == "SUCCEEDED":
            print(f"Skipping {base_filename}, already completed.")
            continue

    # 2. Wait for slot
    while get_active_jobs_count() >= MAX_CONCURRENT_JOBS:
        print("Waiting for slot...")
        time.sleep(300)

    # 3. Start Tuning (Sync=False to prevent widget error)
    gcs_train_path = upload_to_gcs(file, f"{language_name}/tuning/{base_filename}.jsonl")
    
    tuning_job = sft.train(
        source_model="gemini-2.5-flash",
        train_dataset=gcs_train_path,
        validation_dataset=test_set_uri,
        tuned_model_display_name=f"tuned-{base_filename}",
        # Hyperparameters
        epochs=3,
        adapter_size=4,

    )

    # 4. Wait for this specific job to finish so we can run inference
    print(f"Tuning {base_filename}... Job ID: {tuning_job.resource_name}")
    tuning_job.wait() # This blocks the script until the model is ready

    tuned_model_resource = tuning_job.tuned_model_name 
    model = aiplatform.Model(tuned_model_resource)

    # 1. Initialize the Evaluation Task
    eval_task = EvalTask(
        dataset=test_set_uri,  # Uses your existing test set
        metrics=["comet", "bleu"],
        experiment=f"eval-{base_filename}"
    )

    # 2. Run evaluation against your new tuned model
    # This uses the Gen AI Evaluation service
    eval_result = eval_task.evaluate(
        model=model,
        prompt_template=f"Translate to english from {language_name}: {{source}}"
    )

    # 3. Print the summary results
    print(f"Results for {base_filename}: {eval_result.summary_metrics}")
    
    # 5. Run Batch Prediction on the 500 sentences
    # Get the model resource from the tuning job
    
    
    print(f"Starting batch prediction for {base_filename} using model {tuned_model_resource}...")
    batch_job = model.batch_predict(
        job_display_name=f"eval-{base_filename}",
        gcs_source=TEST_SET_GCS_INFERENCE, # Must be in the "request" wrapper format
        gcs_destination_prefix=f"gs://{bucket_name}/eval_results/{base_filename}/",
        instances_format="jsonl",
        predictions_format="jsonl",
        sync=False # Don't block here; we'll move to the next tuning job
    )
    print(f"Started batch prediction job: {batch_job.resource_name}")
    # 6. Final Logging
    new_entry = {
        "batch_name": base_filename, 
        "job_id": tuning_job.resource_name, 
        "status": "SUCCEEDED", 
        "tuned_model_id": tuned_model_resource,
        "batch_prediction_id": batch_job.resource_name
    }
    df_tracking = pd.concat([pd.read_csv(TRACKING_FILE), pd.DataFrame([new_entry])], ignore_index=True)
    df_tracking.to_csv(TRACKING_FILE, index=False)

Processing file: kalamang_all_batch_prompts_ADJ-NUM_400_rule_batch_10
Creating SupervisedTuningJob
SupervisedTuningJob created. Resource name: projects/920145238348/locations/europe-west1/tuningJobs/976611310399651840
To use this SupervisedTuningJob in another session:
tuning_job = sft.SupervisedTuningJob('projects/920145238348/locations/europe-west1/tuningJobs/976611310399651840')
View Tuning Job:
https://console.cloud.google.com/vertex-ai/generative/language/locations/europe-west1/tuning/tuningJob/976611310399651840?project=920145238348


ImportError: cannot import name 'display' from 'IPython.core.display' (C:\Users\Varun\AppData\Roaming\Python\Python313\site-packages\IPython\core\display.py)